# Colab gpu kernel playground

This notebook writes a tiny CUDA kernel, compiles it, and runs it on the Colab GPU.


In [1]:
%%writefile custom_kernel.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void matMul(float* a, float* b, float* c, int m, int k, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < m && col < n) {
        float t = 0.0f;

        for (int i = 0; i < k; ++i) {
            t += a[row * k + i] * b[i * n + col];
        }

        c[row * n + col] = t;
    }
}

int main() {
    const int m = 2;
    const int k = 3;
    const int n = 4;

    const int aSize = m * k;
    const int bSize = k * n;
    const int cSize = m * n;

    float hA[aSize] = {
        1, 2, 3,
        4, 5, 6
    };

    float hB[bSize] = {
        7, 8, 9, 10,
        11, 12, 13, 14,
        15, 16, 17, 18
    };

    float hC[cSize];

    float *dA, *dB, *dC;

    cudaMalloc((void**)&dA, aSize * sizeof(float));
    cudaMalloc((void**)&dB, bSize * sizeof(float));
    cudaMalloc((void**)&dC, cSize * sizeof(float));

    cudaMemcpy(dA, hA, aSize * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, bSize * sizeof(float), cudaMemcpyHostToDevice);

    dim3 blockSize(2, 2);
    dim3 gridSize((n + blockSize.x - 1) / blockSize.x,
                  (m + blockSize.y - 1) / blockSize.y);

    matMul<<<gridSize, blockSize>>>(dA, dB, dC, m, k, n);
    cudaDeviceSynchronize();

    cudaMemcpy(hC, dC, cSize * sizeof(float), cudaMemcpyDeviceToHost);

    std::cout << "Matrix A (" << m << "x" << k << "):" << std::endl;
    for (int row = 0; row < m; row++) {
        for (int col = 0; col < k; col++) {
            std::cout << hA[row * k + col] << "\t";
        }
        std::cout << std::endl;
    }

    std::cout << std::endl;
    std::cout << "Matrix B (" << k << "x" << n << "):" << std::endl;
    for (int row = 0; row < k; row++) {
        for (int col = 0; col < n; col++) {
            std::cout << hB[row * n + col] << "\t";
        }
        std::cout << std::endl;
    }

    std::cout << std::endl;
    std::cout << "Matrix C = A x B (" << m << "x" << n << "):" << std::endl;
    for (int row = 0; row < m; row++) {
        for (int col = 0; col < n; col++) {
            std::cout << hC[row * n + col] << "\t";
        }
        std::cout << std::endl;
    }

    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);

    return 0;
}

Writing custom_kernel.cu


In [2]:
!nvidia-smi

Thu Apr 30 20:24:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!nvcc custom_kernel.cu -o custom_kernel

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./custom_kernel

Matrix A (2x3):
1	2	3	
4	5	6	

Matrix B (3x4):
7	8	9	10	
11	12	13	14	
15	16	17	18	

Matrix C = A x B (2x4):
74	80	86	92	
173	188	203	218	


In [5]:
!ncu ./custom_kernel

==PROF== Connected to process 16600 (/content/custom_kernel)
==PROF== Profiling "matMul" - 0: 0%....50%....100% - 10 passes
Matrix A (2x3):
1	2	3	
4	5	6	

Matrix B (3x4):
7	8	9	10	
11	12	13	14	
15	16	17	18	

Matrix C = A x B (2x4):
74	80	86	92	
173	188	203	218	
==PROF== Disconnected from process 16600
[16600] custom_kernel@127.0.0.1
  matMul(float *, float *, float *, int, int, int) (2, 1, 1)x(2, 2, 1), Context 1, Stream 7, Device 0, CC 8.0
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         1.19
    SM Frequency                    Ghz         1.08
    Elapsed Cycles                cycle        3,942
    Memory Throughput                 %         0.47
    DRAM Throughput                   %         0.11
    Duration                         us         3.65
    L1/TEX Cache Throughput        